In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ------------------ CONFIG ------------------
benchmarks = ["private_enterprise", 'social_media_cloud', 'commercial_cloud', 'university']
loads = [i for i in range(1, 10)]

input_dir = "/home/hsd/workspace/trafpy/examples/comparison_generator/final_data"

# ✅ Three CONGA versions (same filenames, different folders)
outputs = {
    "base": "/home/hsd/workspace/ns3-load-balance/results",
    "v2": "/home/hsd/workspace/ns3-load-balance/results_powerof2",
    "v3": "/home/hsd/workspace/ns3-load-balance/results_powerof2_v2",
}

save_dir = "/home/hsd/workspace/ns3-load-balance/results_powerof2_v1_v2_comparison/analysis_results"
Path(save_dir).mkdir(parents=True,exist_ok=True)

ip_pattern = r'^\d+\.\d+\.\d+\.\d+$'
summary_results = {}

# ------------------ FUNCTIONS ------------------
def clean(df):
    return df[
        df['Src'].str.match(ip_pattern, na=False) &
        df['Dest'].str.match(ip_pattern, na=False)
    ].copy()

def ip_to_node(ip):
    last = int(ip.split('.')[-1])
    last1 = int(ip.split('.')[-2])

    leafcount = 2
    leaf = 0
    if last1 == 2:
        leaf = 1

    return ((last // 2) - 1 + (leaf * leafcount))

def add_time(df):
    df['start_time'] = (
        df['TimeFirstTxPacket']
        .astype(str)
        .str.replace('+','', regex=False)
        .str.replace('ns','', regex=False)
        .astype(float) / 1e9
    )
    return df

flow_columns = [
    "FlowID","Src","Dest","TimeFirstRxPacket","TimeFirstTxPacket",
    "TimeLastRxPacket","TimeLastTxPacket","FCT(s)","TxPackets",
    "RxPackets","LostPackets","LossRate","PDR","LossPercent",
    "TxBytes","RxBytes","Throughput(Kbps)","MeanDelay(ms)",
    "Jitter(ms)","HopCount"
]

def match_df(output_df, input_df, label):
    matches = []

    for _, in_row in input_df.iterrows():
        cand = output_df[
            (output_df.sn == in_row.sn) &
            (output_df.dn == in_row.dn)
        ].copy()

        if len(cand) == 0:
            continue

        cand['time_diff'] = abs(cand['start_time'] - in_row.event_time)
        best = cand.loc[cand['time_diff'].idxmin()]

        combined = {
            'flow_id': in_row.flow_id,
            'sn': in_row.sn,
            'dn': in_row.dn,
            'input_time': in_row.event_time,
            'flow_size': in_row.flow_size,
            f'time_diff_{label}': best['time_diff'],
            f'TxPackets_{label}': best['TxPackets'],
            f'RxPackets_{label}': best['RxPackets'],
            f'TxBytes_calc_{label}': best['TxPackets'] * 1400,
            f'RxBytes_calc_{label}': best['RxPackets'] * 1400,
        }

        for col in flow_columns:
            combined[f"{col}_{label}"] = best[col]

        matches.append(combined)

    return pd.DataFrame(matches)

# ------------------ MAIN LOOP ------------------
for benchmark in benchmarks:
    for load in loads:

        print(f"\n=== {benchmark} | Load 0.{load} ===")

        input_path = f"{input_dir}/{benchmark}_load_{load}.csv"

        # ------------------ LOAD FILES ------------------
        dfs = {}

        try:
            input_df = pd.read_csv(input_path)

            for name, folder in outputs.items():
                file_path = f"{folder}/{benchmark}_load_{load}_Conga.csv"
                dfs[name] = pd.read_csv(file_path, nrows=12000)

        except Exception as e:
            print("Skipping:", e)
            continue

        # ------------------ CLEAN + PREPROCESS ------------------
        for name in dfs:
            dfs[name] = add_time(clean(dfs[name]))
            dfs[name]['sn'] = dfs[name]['Src'].apply(ip_to_node)
            dfs[name]['dn'] = dfs[name]['Dest'].apply(ip_to_node)

        # ------------------ MATCH ------------------
        matched = {}
        for name in dfs:
            matched[name] = match_df(dfs[name], input_df, name)

        # ------------------ MERGE (COMMON FLOWS) ------------------
        keys = ['flow_id', 'sn', 'dn', 'input_time', 'flow_size']

        final_df = matched["base"]
        for name in ["v2", "v3"]:
            final_df = pd.merge(final_df, matched[name], on=keys, how='inner')

        if len(final_df) == 0:
            print("No matches, skipping")
            continue

        # ------------------ DIFFS VS BASE ------------------
        variants = ["v2", "v3"]

        for v in variants:
            final_df[f'fct_diff_{v}'] = (
                final_df[f'FCT(s)_{v}'] - final_df['FCT(s)_base']
            )

            final_df[f'norm_fct_{v}'] = (
                final_df[f'FCT(s)_{v}'] / final_df['FCT(s)_base']
            )

        # ------------------ SAVE CSV ------------------
        csv_path = f"{save_dir}/{benchmark}_load_{load}_full.csv"
        final_df.to_csv(csv_path, index=False)

        # ------------------ STATS ------------------
        base_avg = final_df['FCT(s)_base'].mean()
        print("Base:", base_avg)

        for v in variants:
            avg = final_df[f'FCT(s)_{v}'].mean()
            print(f"{v}:", avg, "| diff:", avg - base_avg)

        # ------------------ STORE SUMMARY ------------------
        if benchmark not in summary_results:
            summary_results[benchmark] = {
                'loads': [],
                'base': [],
                'v2': [],
                'v3': []
            }

        summary_results[benchmark]['loads'].append(load / 10)
        summary_results[benchmark]['base'].append(base_avg)

        for v in variants:
            summary_results[benchmark][v].append(
                final_df[f'FCT(s)_{v}'].mean()
            )

        # ------------------ SCATTER (vs BASE) ------------------
        for v in variants:

            plt.figure()

            plt.scatter(
                final_df['FCT(s)_base'],
                final_df[f'FCT(s)_{v}'],
                alpha=0.5
            )

            max_val = max(
                final_df['FCT(s)_base'].max(),
                final_df[f'FCT(s)_{v}'].max()
            )

            plt.plot([0, max_val], [0, max_val], linestyle='--')

            plt.xlabel("Base Conga FCT")
            plt.ylabel(f"{v} FCT")
            plt.title(f"{benchmark} Load 0.{load}")

            plt.grid()
            plt.savefig(f"{save_dir}/{benchmark}_load_{load}_{v}_vs_base.png")
            plt.close()

        # ------------------ HISTOGRAM ------------------
        for v in variants:

            plt.figure()
            plt.hist(final_df[f'fct_diff_{v}'], bins=30)

            plt.xlabel(f"FCT Diff ({v} - base)")
            plt.ylabel("Count")
            plt.title(f"{benchmark} Load 0.{load}")

            plt.grid()
            plt.savefig(f"{save_dir}/{benchmark}_load_{load}_{v}_hist.png")
            plt.close()

# ------------------ FINAL LINE PLOTS ------------------
for benchmark in summary_results:

    plt.figure()

    for name in ['base', 'v2', 'v3']:
        plt.plot(
            summary_results[benchmark]['loads'],
            summary_results[benchmark][name],
            marker='o',
            label=name
        )

    plt.xlabel("Load")
    plt.ylabel("Average FCT")
    plt.title(f"{benchmark} Avg FCT vs Load")
    plt.legend()
    plt.grid()

    plt.savefig(f"{save_dir}/{benchmark}_avg_fct_vs_load.png")
    plt.close()

print("\n DONE: 3-way CONGA comparison (vs base) completed!")


=== private_enterprise | Load 0.1 ===
Base: 0.006300437166666667
v2: 0.006300819333333333 | diff: 3.8216666666587157e-07
v3: 0.006300819333333333 | diff: 3.8216666666587157e-07

=== private_enterprise | Load 0.2 ===
Base: 0.009051249833333332
v2: 0.009153848 | diff: 0.0001025981666666672
v3: 0.009153848 | diff: 0.0001025981666666672

=== private_enterprise | Load 0.3 ===
Base: 0.012651596166666666
v2: 0.012918194333333334 | diff: 0.00026659816666666815
v3: 0.012918194333333334 | diff: 0.00026659816666666815

=== private_enterprise | Load 0.4 ===
Base: 0.013777815833333335
v2: 0.013776019166666667 | diff: -1.7966666666681397e-06
v3: 0.013776019166666667 | diff: -1.7966666666681397e-06

=== private_enterprise | Load 0.5 ===
Base: 0.019427395833333333
v2: 0.019598856333333334 | diff: 0.0001714605000000015
v3: 0.019598856333333334 | diff: 0.0001714605000000015

=== private_enterprise | Load 0.6 ===
Base: 0.016852931666666664
v2: 0.017616319833333335 | diff: 0.0007633881666666703
v3: 0.017

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

# ------------------ CONFIG ------------------
save_dir = Path("/home/hsd/workspace/ns3-load-balance/results_powerof2_v1_v2_comparison/analysis_results/")
graph_dir = save_dir / "graph"
graph_dir.mkdir(parents=True, exist_ok=True)

benchmarks = ["private_enterprise", 'social_media_cloud', 'commercial_cloud', 'university']
loads = [i for i in range(1, 10)]

SMALL = 100 * 1024        # 100 KB
LARGE = 1 * 1024 * 1024   # 1 MB

variants = ["v2", "v3"]

# ------------------ ANALYSIS ------------------
for benchmark in benchmarks:

    print(f"\n===== {benchmark} =====")

    load_vals = []

    # SMALL
    small_base = []
    small_v2 = []
    small_v3 = []

    # LARGE
    large_base = []
    large_v2 = []
    large_v3 = []

    # NORMALIZED (IMPORTANT)
    small_norm_v2 = []
    small_norm_v3 = []

    large_norm_v2 = []
    large_norm_v3 = []

    for load in loads:

        csv_path = save_dir / f"{benchmark}_load_{load}_full.csv"

        if not csv_path.exists():
            print(f"Missing: {csv_path}")
            continue

        df = pd.read_csv(csv_path)

        if df.empty:
            continue

        # ------------------ SPLIT FLOWS ------------------
        small_df = df[df['flow_size'] < SMALL]
        large_df = df[df['flow_size'] > LARGE]

        # ------------------ SMALL FLOWS ------------------
        if not small_df.empty:
            base_mean = small_df['FCT(s)_base'].mean()
            small_base.append(base_mean)

            v2_mean = small_df['FCT(s)_v2'].mean()
            v3_mean = small_df['FCT(s)_v3'].mean()

            small_v2.append(v2_mean)
            small_v3.append(v3_mean)

            # normalized
            small_norm_v2.append(v2_mean / base_mean)
            small_norm_v3.append(v3_mean / base_mean)

        else:
            small_base.append(np.nan)
            small_v2.append(np.nan)
            small_v3.append(np.nan)
            small_norm_v2.append(np.nan)
            small_norm_v3.append(np.nan)

        # ------------------ LARGE FLOWS ------------------
        if not large_df.empty:
            base_mean = large_df['FCT(s)_base'].mean()
            large_base.append(base_mean)

            v2_mean = large_df['FCT(s)_v2'].mean()
            v3_mean = large_df['FCT(s)_v3'].mean()

            large_v2.append(v2_mean)
            large_v3.append(v3_mean)

            # normalized
            large_norm_v2.append(v2_mean / base_mean)
            large_norm_v3.append(v3_mean / base_mean)

        else:
            large_base.append(np.nan)
            large_v2.append(np.nan)
            large_v3.append(np.nan)
            large_norm_v2.append(np.nan)
            large_norm_v3.append(np.nan)

        load_vals.append(load / 10)

    # ================== PLOT: SMALL FLOWS ==================
    plt.figure()

    plt.plot(load_vals, small_base, marker='o', label='Base')
    plt.plot(load_vals, small_v2, marker='s', label='v2')
    plt.plot(load_vals, small_v3, marker='^', label='v3')

    plt.xlabel("Network Load")
    plt.ylabel("Average FCT (s)")
    plt.title(f"{benchmark}: Small Flows (<100KB)")
    plt.legend()
    plt.grid()

    out_path = graph_dir / f"{benchmark}_SMALL_vs_load.png"
    plt.savefig(out_path)
    plt.close()

    print(f"Saved: {out_path}")

    # ================== PLOT: LARGE FLOWS ==================
    plt.figure()

    plt.plot(load_vals, large_base, marker='o', label='Base')
    plt.plot(load_vals, large_v2, marker='s', label='v2')
    plt.plot(load_vals, large_v3, marker='^', label='v3')

    plt.xlabel("Network Load")
    plt.ylabel("Average FCT (s)")
    plt.title(f"{benchmark}: Large Flows (>1MB)")
    plt.legend()
    plt.grid()

    out_path = graph_dir / f"{benchmark}_LARGE_vs_load.png"
    plt.savefig(out_path)
    plt.close()

    print(f"Saved: {out_path}")

    # ================== NORMALIZED SMALL ==================
    plt.figure()

    plt.plot(load_vals, small_norm_v2, marker='s', label='v2 / base')
    plt.plot(load_vals, small_norm_v3, marker='^', label='v3 / base')

    plt.axhline(1.0, linestyle='--')  # baseline

    plt.xlabel("Network Load")
    plt.ylabel("Normalized FCT")
    plt.title(f"{benchmark}: Small Flows Normalized")
    plt.legend()
    plt.grid()

    out_path = graph_dir / f"{benchmark}_SMALL_normalized.png"
    plt.savefig(out_path)
    plt.close()

    print(f"Saved: {out_path}")

    # ================== NORMALIZED LARGE ==================
    plt.figure()

    plt.plot(load_vals, large_norm_v2, marker='s', label='v2 / base')
    plt.plot(load_vals, large_norm_v3, marker='^', label='v3 / base')

    plt.axhline(1.0, linestyle='--')

    plt.xlabel("Network Load")
    plt.ylabel("Normalized FCT")
    plt.title(f"{benchmark}: Large Flows Normalized")
    plt.legend()
    plt.grid()

    out_path = graph_dir / f"{benchmark}_LARGE_normalized.png"
    plt.savefig(out_path)
    plt.close()

    print(f"Saved: {out_path}")

print("\n Done: Small vs Large + Normalized comparison (vs base)")


===== private_enterprise =====
Saved: /home/hsd/workspace/ns3-load-balance/results_powerof2_v1_v2_comparison/analysis_results/graph/private_enterprise_SMALL_vs_load.png
Saved: /home/hsd/workspace/ns3-load-balance/results_powerof2_v1_v2_comparison/analysis_results/graph/private_enterprise_LARGE_vs_load.png
Saved: /home/hsd/workspace/ns3-load-balance/results_powerof2_v1_v2_comparison/analysis_results/graph/private_enterprise_SMALL_normalized.png
Saved: /home/hsd/workspace/ns3-load-balance/results_powerof2_v1_v2_comparison/analysis_results/graph/private_enterprise_LARGE_normalized.png

===== social_media_cloud =====
Saved: /home/hsd/workspace/ns3-load-balance/results_powerof2_v1_v2_comparison/analysis_results/graph/social_media_cloud_SMALL_vs_load.png
Saved: /home/hsd/workspace/ns3-load-balance/results_powerof2_v1_v2_comparison/analysis_results/graph/social_media_cloud_LARGE_vs_load.png
Saved: /home/hsd/workspace/ns3-load-balance/results_powerof2_v1_v2_comparison/analysis_results/graph/s

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

# ------------------ CONFIG ------------------
save_dir = Path("/home/hsd/workspace/ns3-load-balance/results_powerof2_v1_v2_comparison/analysis_results/")
graph_dir = save_dir / "graph"
graph_dir.mkdir(parents=True, exist_ok=True)

benchmarks = ["private_enterprise", 'social_media_cloud', 'commercial_cloud', 'university']
loads = [i for i in range(1, 10)]

SMALL = 100 * 1024        # 100 KB
LARGE = 1 * 1024 * 1024   # 1 MB

variants = ["v2", "v3"]

# ------------------ MAIN ------------------
for benchmark in benchmarks:

    print(f"\n===== {benchmark} =====")

    small_base, small_v2, small_v3 = [], [], []
    large_base, large_v2, large_v3 = [], [], []
    load_vals = []

    for load in loads:

        csv_path = save_dir / f"{benchmark}_load_{load}_full.csv"
        if not csv_path.exists():
            continue

        df = pd.read_csv(csv_path)
        if len(df) == 0:
            continue

        # ------------------ SPLIT ------------------
        small_df = df[df['flow_size'] < SMALL]
        large_df = df[df['flow_size'] > LARGE]

        # SMALL
        if len(small_df) > 0:
            small_base.append(small_df['FCT(s)_base'].mean())
            small_v2.append(small_df['FCT(s)_v2'].mean())
            small_v3.append(small_df['FCT(s)_v3'].mean())
        else:
            small_base.append(np.nan)
            small_v2.append(np.nan)
            small_v3.append(np.nan)

        # LARGE
        if len(large_df) > 0:
            large_base.append(large_df['FCT(s)_base'].mean())
            large_v2.append(large_df['FCT(s)_v2'].mean())
            large_v3.append(large_df['FCT(s)_v3'].mean())
        else:
            large_base.append(np.nan)
            large_v2.append(np.nan)
            large_v3.append(np.nan)

        load_vals.append(load / 10)

        # ================== COLORED SCATTER (vs BASE) ==================
        for v in variants:

            plt.figure(figsize=(6, 6))

            # Masks
            small_mask = df['flow_size'] < SMALL
            large_mask = df['flow_size'] > LARGE
            mid_mask = ~(small_mask | large_mask)

            # Small
            plt.scatter(
                df.loc[small_mask, 'FCT(s)_base'],
                df.loc[small_mask, f'FCT(s)_{v}'],
                alpha=0.5, label='Small (<100KB)'
            )

            # Medium
            plt.scatter(
                df.loc[mid_mask, 'FCT(s)_base'],
                df.loc[mid_mask, f'FCT(s)_{v}'],
                alpha=0.5, label='Medium'
            )

            # Large
            plt.scatter(
                df.loc[large_mask, 'FCT(s)_base'],
                df.loc[large_mask, f'FCT(s)_{v}'],
                alpha=0.5, label='Large (>1MB)'
            )

            # Diagonal
            max_val = max(
                df['FCT(s)_base'].max(),
                df[f'FCT(s)_{v}'].max()
            )

            plt.plot([0, max_val], [0, max_val], linestyle='--')

            plt.xlabel("Base Conga FCT")
            plt.ylabel(f"{v} FCT")
            plt.title(f"{benchmark} Load 0.{load} ({v} vs Base, Size-colored)")
            plt.legend()
            plt.grid()

            # Save
            out_path = graph_dir / f"{benchmark}_load_{load}_{v}_colored_scatter.png"
            plt.savefig(out_path)
            plt.close()

            print(f"Saved: {out_path}")

print("\n Done: Colored scatter plots (vs base) completed!")


===== private_enterprise =====
Saved: /home/hsd/workspace/ns3-load-balance/results_powerof2_v1_v2_comparison/analysis_results/graph/private_enterprise_load_1_v2_colored_scatter.png
Saved: /home/hsd/workspace/ns3-load-balance/results_powerof2_v1_v2_comparison/analysis_results/graph/private_enterprise_load_1_v3_colored_scatter.png
Saved: /home/hsd/workspace/ns3-load-balance/results_powerof2_v1_v2_comparison/analysis_results/graph/private_enterprise_load_2_v2_colored_scatter.png
Saved: /home/hsd/workspace/ns3-load-balance/results_powerof2_v1_v2_comparison/analysis_results/graph/private_enterprise_load_2_v3_colored_scatter.png
Saved: /home/hsd/workspace/ns3-load-balance/results_powerof2_v1_v2_comparison/analysis_results/graph/private_enterprise_load_3_v2_colored_scatter.png
Saved: /home/hsd/workspace/ns3-load-balance/results_powerof2_v1_v2_comparison/analysis_results/graph/private_enterprise_load_3_v3_colored_scatter.png
Saved: /home/hsd/workspace/ns3-load-balance/results_powerof2_v1_v2_c

In [6]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

# ------------------ CONFIG ------------------
save_dir = Path("/home/hsd/workspace/ns3-load-balance/results_powerof2_v1_v2_comparison/analysis_results/")
graph_dir = save_dir / "graph/fct/normalised"
graph_dir.mkdir(parents=True, exist_ok=True)

benchmarks = ["private_enterprise", 'social_media_cloud', 'commercial_cloud', 'university']
loads = [i for i in range(1, 10)]

SMALL = 100 * 1024        # 100 KB
LARGE = 1 * 1024 * 1024   # 1 MB

# =========================================================
# MAIN
# =========================================================
for benchmark in benchmarks:

    print(f"\n===== {benchmark} =====")

    load_vals = []

    # store normalized values
    small_base = []
    small_v2 = []
    small_v3 = []

    large_base = []
    large_v2 = []
    large_v3 = []

    for load in loads:

        csv_path = save_dir / f"{benchmark}_load_{load}_full.csv"
        if not csv_path.exists():
            continue

        df = pd.read_csv(csv_path)
        if df.empty:
            continue

        # avoid divide-by-zero
        df = df[df['FCT(s)_base'] > 0]

        # ------------------ NORMALIZATION ------------------
        df['norm_base'] = 1.0
        df['norm_v2'] = df['FCT(s)_v2'] / df['FCT(s)_base']
        df['norm_v3'] = df['FCT(s)_v3'] / df['FCT(s)_base']

        # ------------------ SPLIT ------------------
        small_df = df[df['flow_size'] < SMALL]
        large_df = df[df['flow_size'] > LARGE]

        # SMALL
        if len(small_df) > 0:
            small_base.append(1.0)
            small_v2.append(small_df['norm_v2'].mean())
            small_v3.append(small_df['norm_v3'].mean())
        else:
            small_base.append(np.nan)
            small_v2.append(np.nan)
            small_v3.append(np.nan)

        # LARGE
        if len(large_df) > 0:
            large_base.append(1.0)
            large_v2.append(large_df['norm_v2'].mean())
            large_v3.append(large_df['norm_v3'].mean())
        else:
            large_base.append(np.nan)
            large_v2.append(np.nan)
            large_v3.append(np.nan)

        load_vals.append(load / 10)

    # ================== SMALL FLOWS ==================
    plt.figure(figsize=(8,6), dpi=120)

    plt.plot(load_vals, small_base, linestyle='--', label='Base (1.0)')
    plt.plot(load_vals, small_v2, marker='o', label='v2 / base')
    plt.plot(load_vals, small_v3, marker='^', label='v3 / base')

    plt.xlabel("Network Load")
    plt.ylabel("Normalized FCT")
    plt.title(f"{benchmark}: Small Flows (<100KB)")
    plt.legend()
    plt.grid()

    plt.ylim(0, 1.6)
    plt.yticks(np.arange(0, 1.6 + 0.2, 0.2))

    out_path = graph_dir / f"{benchmark}_SMALL_relative_vs_load.png"
    plt.savefig(out_path)
    plt.close()

    print(f"Saved: {out_path}")

    # ================== LARGE FLOWS ==================
    plt.figure(figsize=(8,6), dpi=120)

    plt.plot(load_vals, large_base, linestyle='--', label='Base (1.0)')
    plt.plot(load_vals, large_v2, marker='o', label='v2 / base')
    plt.plot(load_vals, large_v3, marker='^', label='v3 / base')

    plt.xlabel("Network Load")
    plt.ylabel("Normalized FCT")
    plt.title(f"{benchmark}: Large Flows (>1MB)")
    plt.legend()
    plt.grid()

    plt.ylim(0, 1.2)
    plt.yticks(np.arange(0, 1.2 + 0.2, 0.2))

    out_path = graph_dir / f"{benchmark}_LARGE_relative_vs_load.png"
    plt.savefig(out_path)
    plt.close()

    print(f"Saved: {out_path}")

print("\n Done: 3-way normalized (vs base) graphs generated")


===== private_enterprise =====
Saved: /home/hsd/workspace/ns3-load-balance/results_powerof2_v1_v2_comparison/analysis_results/graph/fct/normalised/private_enterprise_SMALL_relative_vs_load.png
Saved: /home/hsd/workspace/ns3-load-balance/results_powerof2_v1_v2_comparison/analysis_results/graph/fct/normalised/private_enterprise_LARGE_relative_vs_load.png

===== social_media_cloud =====
Saved: /home/hsd/workspace/ns3-load-balance/results_powerof2_v1_v2_comparison/analysis_results/graph/fct/normalised/social_media_cloud_SMALL_relative_vs_load.png
Saved: /home/hsd/workspace/ns3-load-balance/results_powerof2_v1_v2_comparison/analysis_results/graph/fct/normalised/social_media_cloud_LARGE_relative_vs_load.png

===== commercial_cloud =====
Saved: /home/hsd/workspace/ns3-load-balance/results_powerof2_v1_v2_comparison/analysis_results/graph/fct/normalised/commercial_cloud_SMALL_relative_vs_load.png
Saved: /home/hsd/workspace/ns3-load-balance/results_powerof2_v1_v2_comparison/analysis_results/gra